<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Implement a Multi-Agent Workflow_sol</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Exercise: Implement a Multi-Agent Workflow


**Purpose:**  
Build a structured **multi-agent workflow** using a Planner Agent, Worker Agents, and Result Aggregation.

**Real-Life Scenario:**  
You are building a **Personal Finance Research Team** that helps users evaluate investment opportunities in AI and tech companies (Microsoft, Google, Amazon, NVIDIA, IBM).

The system should:
- Break down the user query into tasks (Planner)
- Execute research tasks in parallel (Workers)
- Combine results into a final investment report (Aggregator)

## 1. Setup

In [ ]:
# Run this first
!pip install -q "langgraph" "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*"  --force-reinstall
from datetime import datetime
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1

## 2. Tools (Already Provided)

You can use these tools in your agents.

In [ ]:
from langchain.tools import tool

@tool
def get_company_info(company: str) -> str:
    """Fetch company overview, AI projects, profit drivers, and SDG alignment."""
    data = {
        "microsoft": "Leads in cloud AI (Azure, Copilot). Profit driver: Cloud + Enterprise. SDG: 9 and 13.",
        "nvidia": "Dominates AI hardware with GPUs. Main projects: CUDA, DGX. Profit driver: Data center GPUs. SDG: 9.",
        "google": "Excels in search AI and Gemini. Profit driver: Advertising + Cloud. SDG: 9 and 13.",
        "amazon": "Leads in e-commerce AI and AWS. Main projects: Bedrock, SageMaker. SDG: 9 and 12."
    }
    return data.get(company.lower(), f"Information for {company} not available.")

## 3. Your Task

**Implement a Multi-Agent Workflow with the following agents:**

1. **Planner Agent**  
   - Takes the user query and breaks it into sub-tasks.

2. **Worker Agents**  
   - Execute the sub-tasks in parallel using tools.

3. **Aggregator Agent**  
   - Combines all worker results into a final investment report.

**Requirements:**
- Use LangGraph to build the workflow
- Show clear node-based design (Planner → Workers → Aggregator)
- Run a test query and show the final report

## 4. Implement the Multi-Agent Workflow

**Complete the code below:**

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

class AgentState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]
    sub_tasks: list
    results: list
    current_task_index: int # Added to manage task iteration

def planner_node(state):
    """Planner breaks down the query into sub-tasks and initializes the task index."""
    query = state["messages"][-1]
    # For this example, let's make the sub_tasks directly consumable by the worker
    # which uses get_company_info.
    sub_tasks = [
        "Research Microsoft AI projects and profit drivers",
        "Research NVIDIA AI projects and profit drivers",
        "Research Microsoft SDG alignment",
        "Research NVIDIA SDG alignment",
    ]
    return {
        "messages": state["messages"] + ["Planner created sub-tasks for individual research."],
        "sub_tasks": sub_tasks,
        "current_task_index": 0 # Initialize task index
    }

def worker_node(state):
    """Worker executes one sub-task at a time from the sub_tasks list."""
    current_task_index = state["current_task_index"]
    sub_tasks = state["sub_tasks"]

    if current_task_index < len(sub_tasks):
        task = sub_tasks[current_task_index]

        # Extract company name from the task for the tool call
        company_names = ["microsoft", "nvidia", "google", "amazon"]
        found_company = None
        for c in company_names:
            if c in task.lower():
                found_company = c
                break

        if found_company:
            result = get_company_info(found_company) # Assumes get_company_info is available
        else:
            result = f"Information not directly available for task: '{task}'"
            # In a real scenario, this might trigger another LLM call or a different tool.

        # Update state with new message, result, and increment task index for next iteration
        return {
            "messages": state["messages"] + [f"Worker processed task {current_task_index + 1}/{len(sub_tasks)}: {task}"],
            "results": state.get("results", []) + [result],
            "current_task_index": current_task_index + 1 # Increment for the next task
        }
    else:
        # This case should ideally not be reached if the conditional edge is correct
        return state

def aggregator_node(state):
    """Aggregator combines all results into a final report."""
    if not state["results"]:
        final_report = "No results were generated for the investment report."
    else:
        final_report = "=== FINAL INVESTMENT REPORT ===\n\n"
        final_report += "Here is the consolidated research:\n\n"
        for i, res in enumerate(state["results"]):
            final_report += f"--- Research Item {i+1} ---\n{res}\n\n"

    return {"messages": state["messages"] + [final_report]}

# Define the conditional edge logic
def should_continue(state):
    """Determines if more sub-tasks need to be processed."""
    if state["current_task_index"] < len(state["sub_tasks"]):
        return "continue_processing" # Loop back to worker
    else:
        return "finish_processing"   # Go to aggregator

# Build the graph
workflow = StateGraph(AgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("worker", worker_node)
workflow.add_node("aggregator", aggregator_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "worker") # Planner always goes to the first worker call

# Conditional edge from worker: either loop back to worker or go to aggregator
workflow.add_conditional_edges(
    "worker", # Source node
    should_continue, # The function that decides the next step
    {
        "continue_processing": "worker",    # If more tasks, loop back to the 'worker' node
        "finish_processing": "aggregator" # If all tasks are done, proceed to 'aggregator'
    }
)

workflow.add_edge("aggregator", END)

app = workflow.compile()

print("Multi-Agent Workflow compiled successfully with dynamic looping")

Multi-Agent Workflow compiled successfully with dynamic looping


## 5. Test Your Workflow

Run the following test query after completing the code:

In [ ]:
test_query = "Evaluate Microsoft and NVIDIA for long-term investment. Consider AI projects, profit drivers, and SDG alignment."

initial_state = {"messages": [test_query]}

result = app.invoke(initial_state)

print("=== FINAL INVESTMENT REPORT ===")
print(result["messages"][-1]);

=== FINAL INVESTMENT REPORT ===
=== FINAL INVESTMENT REPORT ===

Here is the consolidated research:

--- Research Item 1 ---
Leads in cloud AI (Azure, Copilot). Profit driver: Cloud + Enterprise. SDG: 9 and 13.

--- Research Item 2 ---
Dominates AI hardware with GPUs. Main projects: CUDA, DGX. Profit driver: Data center GPUs. SDG: 9.

--- Research Item 3 ---
Leads in cloud AI (Azure, Copilot). Profit driver: Cloud + Enterprise. SDG: 9 and 13.

--- Research Item 4 ---
Dominates AI hardware with GPUs. Main projects: CUDA, DGX. Profit driver: Data center GPUs. SDG: 9.




## 6. Reflection

1. How did the Planner Agent help break down the task?
2. What was the benefit of running Worker Agents in parallel?
3. How effective was the Aggregator in combining results?
4. What challenges did you face while building this multi-agent workflow?